# Chapter 3: Tokenizers

Before a transformer sees any text it must be converted into numbers.
**Tokenization** is that conversion step: a raw string goes in, a sequence of
integer token IDs comes out. Different strategies make different trade-offs
between vocabulary size, out-of-vocabulary handling, and computational cost.

This chapter covers six strategies side-by-side, each applied to the same
sample sentences so the differences are directly comparable:

| Strategy | Library | Vocab required? |
|---|---|---|
| Whitespace | (built-in) | No |
| Regex | `re` | No |
| BPE | HuggingFace `tokenizers` | Trained |
| WordPiece | HuggingFace `tokenizers` | Trained |
| SentencePiece | `sentencepiece` | Trained |
| tiktoken | `tiktoken` | Pre-built (cl100k\_base) |

## Shared Corpus and Sample Sentence

The three learning-based tokenizers (BPE, WordPiece, SentencePiece) are trained
on the same small corpus. We then run all six tokenizers on a fixed **probe
sentence** so the outputs are directly comparable.


In [1]:
CORPUS = [
"the quick brown fox jumps over the lazy dog",
"transformers are powerful models for natural language processing",
"tokenization is the first step in any NLP pipeline",
"neural networks learn representations from data",
"attention mechanisms let every token query every other token",
"byte pair encoding merges the most frequent character pairs",
"word piece splits words into subword units using a trained vocabulary",
"sentence piece learns a subword vocabulary directly from raw text",
]

PROBE = "transformers tokenize text into subword units"
print('Corpus sentences:', len(CORPUS))
print('Probe:', PROBE)


Corpus sentences: 8
Probe: transformers tokenize text into subword units


## Whitespace Tokenizer

The simplest possible strategy: split on any whitespace. No vocabulary, no
training, just `str.split()`. Every word is a token, so the vocabulary is
unbounded and any unseen word becomes its own token. Useful for quick
prototyping or when the input is already clean.

In [2]:
from transformer_book_lab.whitespace_tokenizer import WhitespaceTokenizer

ws = WhitespaceTokenizer()
tokens = ws.tokenize(PROBE)
print('Tokens :', tokens)
print('Count  :', len(tokens))


Tokens : ['transformers', 'tokenize', 'text', 'into', 'subword', 'units']
Count  : 6


## Regex Tokenizer

A step up from whitespace: use a **regular expression** to define what counts
as a token. The pattern `\w+` matches word characters (letters, digits,
underscores), which strips punctuation without needing a vocabulary. The caller
controls granularity by choosing the pattern.


In [3]:
from transformer_book_lab.regex_tokenizer import RegexTokenizer

rx = RegexTokenizer(pattern=r'\w+')
tokens = rx.tokenize(PROBE)
print('Pattern:', rx.pattern)
print('Tokens :', tokens)
print('Count  :', len(tokens))

# Demonstrate a finer-grained pattern that keeps apostrophes
rx2 = RegexTokenizer(pattern=r"[\w']+")
print("\nWith [\\w']+ on \"don't stop\":", rx2.tokenize("don't stop"))


Pattern: \w+
Tokens : ['transformers', 'tokenize', 'text', 'into', 'subword', 'units']
Count  : 6

With [\w']+ on "don't stop": ["don't", 'stop']


## BPE Tokenizer (Byte-Pair Encoding)

BPE starts from individual characters (or bytes) and iteratively merges the
most frequent adjacent pair into a new token. Repeating this `vocab_size` times
produces a vocabulary that captures common subwords while keeping rare words
decomposable. GPT-2, RoBERTa, and most modern LLMs use BPE.

Our `BpeTokenizer` wraps HuggingFace `tokenizers` with a `ByteLevel`
pre-tokeniser so every byte is representable and the encode→decode round-trip
is lossless.


In [4]:
from transformer_book_lab.bpe_tokenizer import BpeTokenizer

bpe = BpeTokenizer(corpus=CORPUS, vocab_size=300)
ids    = bpe.encode(PROBE)
decoded = bpe.decode(ids)

print(f'Vocab size : {bpe.vocab_size}')
print(f'Token IDs  : {ids}')
print(f'Decoded    : {decoded!r}')
print(f'Round-trip : {decoded == PROBE}')


Vocab size : 300
Token IDs  : [83, 265, 279, 69, 259, 76, 257, 82, 261, 280, 72, 89, 68, 261, 68, 87, 83, 220, 258, 83, 78, 273, 84, 292, 220, 84, 77, 72, 83, 82]
Decoded    : 'transformers tokenize text into subword units'
Round-trip : True


### How BPE Builds Its Vocabulary

BPE starts from the smallest possible units, individual bytes, and iteratively
builds larger tokens by merging the most frequent adjacent pair. The process runs
until the target `vocab_size` is reached.

**Step-by-step:**
1. Represent every word as a sequence of characters (e.g. `l o w`)
2. Count every adjacent pair across the corpus (`lo` → 3 times, `ow` → 5 times, …)
3. Merge the most frequent pair into a new token (`o w` → `ow`)
4. Repeat: each merge adds one token to the vocabulary

**Why byte-level?** Our wrapper uses a `ByteLevel` pre-tokenizer (the same trick
GPT-2 uses). Every possible input byte has a corresponding initial token, so
**no character can ever be out of vocabulary,** the encode/decode round-trip is
always lossless.

In [12]:
from collections import Counter

def _pairs(vocab):
    counts = Counter()
    for word, freq in vocab.items():
        syms = word.split()
        for a, b in zip(syms, syms[1:]):
            counts[(a, b)] += freq
    return counts

def _merge(pair, vocab):
    joined = ''.join(pair)
    return {w.replace(' '.join(pair), joined): f for w, f in vocab.items()}

# Character-level representation of a tiny corpus
tiny = {"l o w": 2, "l o w e r": 1, "n e w e s t": 3, "w i d e s t": 2}

print("Initial (character-level):")
for w, f in tiny.items():
    print(f"  {f}×  {w}")

for step in range(6):
    pairs = _pairs(tiny)
    best = max(pairs, key=pairs.get)  # ties go to the first key in iteration order

    print(f"\n--- Step {step + 1} ---")
    print("Pair frequencies (descending):")
    for pair, count in sorted(pairs.items(), key=lambda x: -x[1]):
        marker = " ← selected" if pair == best else ("  ← tied" if count == pairs[best] else "")
        print(f"  {count}×  {pair[0]!r} + {pair[1]!r}{marker}")

    tiny = _merge(best, tiny)
    print(f"Merge: {best[0]!r} + {best[1]!r} → {''.join(best)!r}")
    print("Vocabulary:")
    for w, f in tiny.items():
        print(f"  {f}×  {w}")


Initial (character-level):
  2×  l o w
  1×  l o w e r
  3×  n e w e s t
  2×  w i d e s t

--- Step 1 ---
Pair frequencies (descending):
  5×  'e' + 's' ← selected
  5×  's' + 't'  ← tied
  4×  'w' + 'e'
  3×  'l' + 'o'
  3×  'o' + 'w'
  3×  'n' + 'e'
  3×  'e' + 'w'
  2×  'w' + 'i'
  2×  'i' + 'd'
  2×  'd' + 'e'
  1×  'e' + 'r'
Merge: 'e' + 's' → 'es'
Vocabulary:
  2×  l o w
  1×  l o w e r
  3×  n e w es t
  2×  w i d es t

--- Step 2 ---
Pair frequencies (descending):
  5×  'es' + 't' ← selected
  3×  'l' + 'o'
  3×  'o' + 'w'
  3×  'n' + 'e'
  3×  'e' + 'w'
  3×  'w' + 'es'
  2×  'w' + 'i'
  2×  'i' + 'd'
  2×  'd' + 'es'
  1×  'w' + 'e'
  1×  'e' + 'r'
Merge: 'es' + 't' → 'est'
Vocabulary:
  2×  l o w
  1×  l o w e r
  3×  n e w est
  2×  w i d est

--- Step 3 ---
Pair frequencies (descending):
  3×  'l' + 'o' ← selected
  3×  'o' + 'w'  ← tied
  3×  'n' + 'e'  ← tied
  3×  'e' + 'w'  ← tied
  3×  'w' + 'est'  ← tied
  2×  'w' + 'i'
  2×  'i' + 'd'
  2×  'd' + 'est'
  1×  'w' + 

In [11]:
# See the actual subword strings our trained BPE tokenizer produces, not just IDs
enc = bpe._tokenizer.encode(PROBE)
print("Probe        :", PROBE)
print("Token strings:", enc.tokens)
print("Token IDs    :", enc.ids)
print()

# Sample of the learned vocabulary (sorted by ID)
vocab = sorted(bpe._tokenizer.get_vocab().items(), key=lambda x: x[1])
print("First 20 vocabulary entries (ID → token):")
for token, idx in vocab[:20]:
    print(f"  {idx:>4}  {token!r}")


Probe        : transformers tokenize text into subword units
Token strings: ['t', 'ra', 'ns', 'f', 'or', 'm', 'er', 's', 'Ġt', 'oken', 'i', 'z', 'e', 'Ġt', 'e', 'x', 't', 'Ġ', 'in', 't', 'o', 'Ġs', 'u', 'bword', 'Ġ', 'u', 'n', 'i', 't', 's']
Token IDs    : [83, 265, 279, 69, 259, 76, 257, 82, 261, 280, 72, 89, 68, 261, 68, 87, 83, 220, 258, 83, 78, 273, 84, 292, 220, 84, 77, 72, 83, 82]

First 20 vocabulary entries (ID → token):
     0  '!'
     1  '"'
     2  '#'
     3  '$'
     4  '%'
     5  '&'
     6  "'"
     7  '('
     8  ')'
     9  '*'
    10  '+'
    11  ','
    12  '-'
    13  '.'
    14  '/'
    15  '0'
    16  '1'
    17  '2'
    18  '3'
    19  '4'


## WordPiece Tokenizer

WordPiece is BERT's tokenizer. Like BPE it builds a subword vocabulary, but
instead of merging the *most frequent* pair it merges the pair that maximises
the likelihood of the training corpus under a unigram language model.

Continuation subwords are prefixed with `##` (e.g. `playing` → `play` + `##ing`),
making the word boundary recoverable at decode time.


In [5]:
from transformer_book_lab.wordpiece_tokenizer import WordPieceTokenizer

wp = WordPieceTokenizer(corpus=CORPUS, vocab_size=200)
ids     = wp.encode(PROBE)
decoded  = wp.decode(ids)

print(f'Vocab size : {wp.vocab_size}')
print(f'Token IDs  : {ids}')
print(f'Decoded    : {decoded!r}')
print(f'Round-trip : {decoded == PROBE}')


Vocab size : 200
Token IDs  : [91, 103, 159, 167, 43, 199, 32, 151, 181, 84, 175, 116, 152, 119]
Decoded    : 'transformers tokenize text into subword units'
Round-trip : True


### How WordPiece Builds Its Vocabulary

WordPiece shares BPE's iterative merge loop but changes the **selection criterion**.
Instead of picking the most frequent pair it picks the pair with the highest
*likelihood score*:

```
score(a, b) = freq(ab) / (freq(a) × freq(b))
```

This rewards pairs that almost always appear together relative to how often each
piece appears alone. A pair that is frequent only because both pieces are globally
common scores lower than a rarer pair that is nearly always seen as a unit.

**The `##` prefix** marks continuation subwords: pieces that do not start a word.
For example, `playing` becomes `play` + `##ing`. This makes word boundaries
explicit in the token stream and lets the decoder reconstruct the original text
without ambiguity. It also means WordPiece tokens are not interchangeable with
BPE tokens even when both produce the same surface string.

**Pre-tokenizer difference**: BPE uses a `ByteLevel` pre-tokenizer (all bytes
representable, no `[UNK]`). WordPiece uses `Whitespace`: it first splits on
spaces, then applies subword splitting within each word. Unseen byte combinations
fall back to `[UNK]`.

In [ ]:
from collections import Counter

def _wp_pairs(vocab):
    pair_counts = Counter()
    token_counts = Counter()
    for word, freq in vocab.items():
        syms = word.split()
        for sym in syms:
            token_counts[sym] += freq
        for a, b in zip(syms, syms[1:]):
            pair_counts[(a, b)] += freq
    return pair_counts, token_counts

def _wp_score(pair, pair_counts, token_counts):
    a, b = pair
    return pair_counts[pair] / (token_counts[a] * token_counts[b])

def _wp_merge(pair, vocab):
    joined = ''.join(pair)
    return {w.replace(' '.join(pair), joined): f for w, f in vocab.items()}

# Same tiny corpus as the BPE trace, different criterion will produce different merges
tiny_wp = {"l o w": 2, "l o w e r": 1, "n e w e s t": 3, "w i d e s t": 2}

print("Initial (character-level):")
for w, f in tiny_wp.items():
    print(f"  {f}×  {w}")

for step in range(6):
    pair_counts, token_counts = _wp_pairs(tiny_wp)
    scores = {pair: _wp_score(pair, pair_counts, token_counts) for pair in pair_counts}
    best = max(scores, key=scores.get)

    print(f"\n--- Step {step + 1} ---")
    print("Pair scores  freq(ab) / (freq(a) × freq(b))  [descending]:")
    for pair, score in sorted(scores.items(), key=lambda x: -x[1]):
        marker = " ← selected" if pair == best else ("  ← tied" if score == scores[best] else "")
        print(f"  {score:.4f}  {pair[0]!r} + {pair[1]!r}  (freq={pair_counts[pair]}){marker}")

    tiny_wp = _wp_merge(best, tiny_wp)
    print(f"Merge: {best[0]!r} + {best[1]!r} → {''.join(best)!r}")
    print("Vocabulary:")
    for w, f in tiny_wp.items():
        print(f"  {f}×  {w}")

In [14]:
# See the actual subword strings with ## continuation markers
enc = wp._tokenizer.encode(PROBE)
print("Probe        :", PROBE)
print("Token strings:", enc.tokens)
print("Token IDs    :", enc.ids)
print()

# Sample of the learned vocabulary
vocab = sorted(wp._tokenizer.get_vocab().items(), key=lambda x: x[1])
print("First 20 vocabulary entries (ID → token):")
for token, idx in vocab[:20]:
    print(f"  {idx:>4}  {token!r}")


Probe        : transformers tokenize text into subword units
Token strings: ['tra', '##ns', '##for', '##mer', '##s', 'tokeniz', '##e', 'te', '##xt', 'in', '##to', 'subword', 'un', '##its']
Token IDs    : [91, 103, 159, 167, 43, 199, 32, 151, 181, 84, 175, 116, 152, 119]

First 20 vocabulary entries (ID → token):
     0  '[UNK]'
     1  'L'
     2  'N'
     3  'P'
     4  'a'
     5  'b'
     6  'c'
     7  'd'
     8  'e'
     9  'f'
    10  'g'
    11  'h'
    12  'i'
    13  'j'
    14  'k'
    15  'l'
    16  'm'
    17  'n'
    18  'o'
    19  'p'


## SentencePiece Tokenizer

SentencePiece treats the input as a raw stream of Unicode characters, spaces
included, so it is **language-agnostic** and works well for languages without
whitespace word boundaries (Chinese, Japanese). It supports both BPE and Unigram
model types; we use BPE here for comparability.

Unlike HuggingFace `tokenizers`, SentencePiece writes a model file during
training. Our wrapper keeps that inside a temporary directory so no files are
left on disk after construction.

In [6]:
from transformer_book_lab.sentencepiece_tokenizer import SentencePieceTokenizer

sp = SentencePieceTokenizer(corpus=CORPUS, vocab_size=50)
ids     = sp.encode(PROBE)
decoded  = sp.decode(ids)

print(f'Vocab size : {sp.vocab_size}')
print(f'Token IDs  : {ids}')
print(f'Decoded    : {decoded!r}')
print(f'Round-trip : {decoded == PROBE}')


Vocab size : 50
Token IDs  : [3, 12, 24, 27, 36, 7, 34, 5, 27, 3, 26, 41, 4, 28, 45, 21, 3, 21, 44, 23, 20, 6, 23, 26, 14, 30, 40, 15, 33, 20, 30, 24, 28, 23, 27]
Decoded    : 'transformers tokenize text into subword units'
Round-trip : True


### How SentencePiece Differs

SentencePiece shares the BPE merge algorithm but changes a more fundamental
assumption: **there is no pre-tokenizer**. The raw Unicode character stream,
spaces included, is fed directly to the learner. This has three consequences:

**`▁` marks word starts, not continuations.**
SentencePiece encodes every space as the special character `▁` (U+2581) and
attaches it to the *following* token. So `" hello"` becomes `▁hello`. Compare
this with the other two approaches:

| Tokenizer | Space encoding | Example |
|---|---|---|
| BPE (ByteLevel) | `Ġ` prefix on space-initial bytes | `Ġhello` |
| WordPiece | via `Whitespace` pre-tokenizer, splits on spaces | `hello` / `##lo` |
| SentencePiece | `▁` prefix on word-initial pieces | `▁hello` |

**Language-agnostic.**
Because there is no assumption that spaces mark word boundaries,
SentencePiece works out-of-the-box on languages without whitespace word
boundaries (Chinese, Japanese, Thai). The same training code runs unchanged.

**Binary model file.**
SentencePiece writes a `.model` file during training. Our wrapper uses a
temporary directory so no files are left on disk after construction, the
trained model is loaded entirely into memory.

In [15]:
# See the actual piece strings — note ▁ marking word-initial pieces
pieces = sp._model.encode(PROBE, out_type=str)
print("Probe        :", PROBE)
print("Pieces       :", pieces)
print("Token IDs    :", sp.encode(PROBE))
print()

# Side-by-side space representation across all three subword tokenizers
bpe_tokens = bpe._tokenizer.encode(PROBE).tokens
wp_tokens  = wp._tokenizer.encode(PROBE).tokens
sp_pieces  = sp._model.encode(PROBE, out_type=str)

print("Space representation across subword tokenizers:")
print(f"  BPE (ByteLevel) : {bpe_tokens}")
print(f"  WordPiece       : {wp_tokens}")
print(f"  SentencePiece   : {sp_pieces}")


Probe        : transformers tokenize text into subword units
Pieces       : ['▁t', 'ra', 'n', 's', 'f', 'or', 'm', 'er', 's', '▁t', 'o', 'k', 'en', 'i', 'z', 'e', '▁t', 'e', 'x', 't', '▁', 'in', 't', 'o', '▁s', 'u', 'b', 'wor', 'd', '▁', 'u', 'n', 'i', 't', 's']
Token IDs    : [3, 12, 24, 27, 36, 7, 34, 5, 27, 3, 26, 41, 4, 28, 45, 21, 3, 21, 44, 23, 20, 6, 23, 26, 14, 30, 40, 15, 33, 20, 30, 24, 28, 23, 27]

Space representation across subword tokenizers:
  BPE (ByteLevel) : ['t', 'ra', 'ns', 'f', 'or', 'm', 'er', 's', 'Ġt', 'oken', 'i', 'z', 'e', 'Ġt', 'e', 'x', 't', 'Ġ', 'in', 't', 'o', 'Ġs', 'u', 'bword', 'Ġ', 'u', 'n', 'i', 't', 's']
  WordPiece       : ['tra', '##ns', '##for', '##mer', '##s', 'tokeniz', '##e', 'te', '##xt', 'in', '##to', 'subword', 'un', '##its']
  SentencePiece   : ['▁t', 'ra', 'n', 's', 'f', 'or', 'm', 'er', 's', '▁t', 'o', 'k', 'en', 'i', 'z', 'e', '▁t', 'e', 'x', 't', '▁', 'in', 't', 'o', '▁s', 'u', 'b', 'wor', 'd', '▁', 'u', 'n', 'i', 't', 's']


In [ ]:
# Language-agnostic property: encode text without whitespace word boundaries
# Our model is only trained on English, so pieces will be granular,
# but crucially there is no [UNK] and no crash, the model degrades gracefully.
samples = [
    ("English",  "hello world"),
    ("Spanish",  "hola mundo"),
    ("Japanese", "こんにちは世界"),
    ("Arabic",   "مرحبا بالعالم"),
]
for lang, text in samples:
    pieces = sp._model.encode(text, out_type=str)
    print(f"{lang:<10} {text!r:<30} → {pieces}")

## tiktoken Tokenizer (`cl100k_base`)

`tiktoken` is OpenAI's production tokenizer library. `cl100k_base` is the
encoding used by GPT-4, GPT-3.5-turbo, and the embeddings API. It has
**100 277 tokens** and uses a ByteLevel BPE trained on a vast multilingual
corpus. No training is needed here, the vocabulary is pre-built and cached
locally after the first download.

This is the most production-realistic tokenizer in this chapter.

In [7]:
from transformer_book_lab.tiktoken_tokenizer import TiktokenTokenizer

tt = TiktokenTokenizer()
ids     = tt.encode(PROBE)
decoded  = tt.decode(ids)

print(f'Vocab size : {tt.vocab_size:,}')
print(f'Token IDs  : {ids}')
print(f'Decoded    : {decoded!r}')
print(f'Round-trip : {decoded == PROBE}')


Vocab size : 100,277
Token IDs  : [4806, 388, 78751, 1495, 1139, 1207, 1178, 8316]
Decoded    : 'transformers tokenize text into subword units'
Round-trip : True


### How tiktoken Works

`tiktoken` differs from the three trained tokenizers in every structural dimension:

**Pre-built vocabulary: no training.**
`cl100k_base` ships with a fixed vocabulary of 100 277 tokens learned by OpenAI
on a massive multilingual corpus. There is no `corpus` argument; the encoding is
downloaded once and cached locally.

**Regex pre-tokenizer.**
Before BPE merges are applied, the text is split by a hand-crafted regex that
enforces consistent boundaries. Numbers are capped at 3 digits per token,
contractions (`'t`, `'re`, `'ve`, …) are kept as units, and punctuation is
separated from letters. This prevents semantically unrelated text from being
merged into the same token.

**Special tokens.**
`cl100k_base` reserves a range of IDs above 100 256 for control tokens such as
`<|endoftext|>` and fill-in-the-middle markers. These are never produced by
normal text encoding, they must be passed explicitly.

**Production coverage.**
This is the encoding used by GPT-4, GPT-3.5-turbo, and the OpenAI embeddings
API. Its 100 k vocabulary means most common English words and subwords map to a
single token, giving shorter sequences, and therefore cheaper attention, than
a smaller trained vocabulary would.

In [17]:
# Token strings for the probe sentence
enc_ids = tt.encode(PROBE)
tokens  = [tt._enc.decode([i]) for i in enc_ids]
print("Probe        :", PROBE)
print("Token strings:", tokens)
print("Token IDs    :", enc_ids)
print()

# The regex pre-tokenizer enforces consistent boundaries before BPE.
# Show how it handles contractions, number length, and punctuation.
examples = [
    ("contraction",  "don't stop"),
    ("numbers",      "12345 items"),
    ("punctuation",  "hello, world!"),
    ("mixed",        "GPT-4 is fast"),
]
print("Pre-tokenizer behaviour on tricky inputs:")
for label, text in examples:
    ids     = tt.encode(text)
    decoded = [tt._enc.decode([i]) for i in ids]
    print(f"  {label:<14} {text!r:<25} → {decoded}")


Probe        : transformers tokenize text into subword units
Token strings: ['transform', 'ers', ' tokenize', ' text', ' into', ' sub', 'word', ' units']
Token IDs    : [4806, 388, 78751, 1495, 1139, 1207, 1178, 8316]

Pre-tokenizer behaviour on tricky inputs:
  contraction    "don't stop"              → ['don', "'t", ' stop']
  numbers        '12345 items'             → ['123', '45', ' items']
  punctuation    'hello, world!'           → ['hello', ',', ' world', '!']
  mixed          'GPT-4 is fast'           → ['G', 'PT', '-', '4', ' is', ' fast']


In [18]:
# Special tokens — reserved IDs outside the regular vocabulary range
print("Special tokens in cl100k_base:")
for token, idx in tt._enc._special_tokens.items():
    print(f"  {idx:>7}  {token!r}")
print()

# Large vocabulary efficiency: most common words map to a single token
words = ["transformer", "tokenization", "attention", "language", "the", "a"]
print("Token count for common words (large vocab → fewer tokens per word):")
for word in words:
    ids     = tt.encode(word)
    decoded = [tt._enc.decode([i]) for i in ids]
    print(f"  {word!r:<16} → {len(ids)} token(s)  {decoded}")


Special tokens in cl100k_base:
   100257  '<|endoftext|>'
   100258  '<|fim_prefix|>'
   100259  '<|fim_middle|>'
   100260  '<|fim_suffix|>'
   100276  '<|endofprompt|>'

Token count for common words (large vocab → fewer tokens per word):
  'transformer'    → 2 token(s)  ['transform', 'er']
  'tokenization'   → 2 token(s)  ['token', 'ization']
  'attention'      → 1 token(s)  ['attention']
  'language'       → 1 token(s)  ['language']
  'the'            → 1 token(s)  ['the']
  'a'              → 1 token(s)  ['a']


## Comparison: All Six Strategies on the Same Input

The table below shows how each tokenizer handles the probe sentence, making
the vocabulary size / token count trade-off visible.


In [19]:
rows = [
    ('Whitespace',    ws.tokenize(PROBE),   'unbounded',       '-'),
    ('Regex (\\w+)',   rx.tokenize(PROBE),   'unbounded',       '-'),
    ('BPE',           bpe.encode(PROBE),     bpe.vocab_size,    'trained'),
    ('WordPiece',     wp.encode(PROBE),      wp.vocab_size,     'trained'),
    ('SentencePiece', sp.encode(PROBE),       sp.vocab_size,     'trained'),
    ('tiktoken',      tt.encode(PROBE),       tt.vocab_size,     'pre-built'),
]

print(f'Probe: {PROBE!r}\n')
print(f'{"Strategy":<16} {"Tokens / IDs":<45} {"Count":>5}  {"Vocab":>8}  {"Training"}')
print('-' * 90)
for name, toks, vocab, training in rows:
    preview = str(toks[:6]) + ('...' if len(toks) > 6 else '')
    print(f'{name:<16} {preview:<45} {len(toks):>5}  {str(vocab):>8}  {training}')


Probe: 'transformers tokenize text into subword units'

Strategy         Tokens / IDs                                  Count     Vocab  Training
------------------------------------------------------------------------------------------
Whitespace       ['transformers', 'tokenize', 'text', 'into', 'subword', 'units']     6  unbounded  -
Regex (\w+)      ['transformers', 'tokenize', 'text', 'into', 'subword', 'units']     6  unbounded  -
BPE              [83, 265, 279, 69, 259, 76]...                   30       300  trained
WordPiece        [91, 103, 159, 167, 43, 199]...                  14       200  trained
SentencePiece    [3, 12, 24, 27, 36, 7]...                        35        50  trained
tiktoken         [4806, 388, 78751, 1495, 1139, 1207]...           8    100277  pre-built


## Key Takeaways

- **Whitespace and regex tokenizers are stateless**: no training, no vocabulary.
  They work as long as word identity is preserved by whitespace, but they cannot
  share representations across morphological variants (`run`/`running`).
- **BPE, WordPiece, and SentencePiece are all subword strategies**: they
  decompose rare words into known pieces, bounding vocabulary size while keeping
  out-of-vocabulary tokens rare.
- **BPE merges by frequency; WordPiece merges by likelihood**: in practice the
  vocabularies look similar, but WordPiece's `##` prefix makes word boundaries
  explicit in the token stream.
- **SentencePiece treats spaces as characters**: this makes it language-agnostic
  and lets it learn tokenization from any raw text without a pre-tokenizer.
- **tiktoken is a fixed, production-grade BPE vocabulary**: no training required,
  100 277 tokens covering essentially all Unicode, deterministic across versions.
- **Vocab size trades off against token count**: a larger vocabulary encodes text
  in fewer tokens (shorter sequences = faster attention), but requires more
  embedding parameters.

## Final Exercise

Train a `BpeTokenizer` and a `WordPieceTokenizer` on the shared `CORPUS` with
`vocab_size=150`. Encode the probe sentence with both and print:

1. The token count for each.
2. Whether the two tokenizers produce the **same** token IDs for the probe.

**Why this matters**: BPE and WordPiece use the same training data and target
vocabulary size but different merge criteria. Comparing their outputs on the same
input reveals whether the criteria matter in practice for small corpora.


In [9]:
from transformer_book_lab.bpe_tokenizer import BpeTokenizer
from transformer_book_lab.wordpiece_tokenizer import WordPieceTokenizer

bpe150 = BpeTokenizer(corpus=CORPUS, vocab_size=150)
wp150  = WordPieceTokenizer(corpus=CORPUS, vocab_size=150)

bpe_ids = bpe150.encode(PROBE)
wp_ids  = wp150.encode(PROBE)

print(f'BPE token count      : {len(bpe_ids)}')
print(f'WordPiece token count: {len(wp_ids)}')
print(f'Same IDs?            : {bpe_ids == wp_ids}')


BPE token count      : 45
WordPiece token count: 22
Same IDs?            : False


### Conclusion

With `vocab_size=150` on the same small corpus, BPE produced **45 tokens** for
the probe sentence while WordPiece produced only **22**,roughly half. The IDs
were also completely different (`Same IDs? False`), for two distinct reasons:

**Why the token counts differ so much.**
WordPiece selects merges that maximise corpus likelihood rather than raw
frequency. On a small corpus this tends to produce longer, more linguistically
coherent subwords earlier in the merge sequence. BPE, by contrast, merges
whatever pair appears most often, on a tiny corpus that often means short
character n-grams, leaving more words fragmented at the same vocab budget.
In other words, the merge criterion matters, especially when training data is
scarce.

**Why the IDs are always different.**
Each tokenizer builds its own vocabulary index independently, so even if two
tokenizers produced the *same* segmentation (e.g., both split `transformers`
into `transform` + `##ers`), the integer IDs assigned to those pieces would
still differ. ID equality is only meaningful *within* one tokenizer's
encode/decode cycle.

**Takeaway for transformer design.**
Choosing BPE vs WordPiece is not just a detail, it affects sequence length,
which directly drives attention cost (O(n²)). On small or domain-specific
corpora, WordPiece's likelihood-based criterion can be noticeably more
efficient than BPE's frequency-based one at the same vocabulary size.
